# 🏭 PumpGuard AI — IoT Demo trên Google Colab

> Chạy toàn bộ hệ thống IoT (MQTT + FastAPI + AI) trực tiếp trên Colab.
> **Không cần cài gì trên máy tính. Không cần tài khoản ngrok.**

## Các bước:
| Bước | Nội dung | Thời gian |
|------|----------|----------|
| 1 | 🔑 Điền Gemini API key | 30 giây |
| 2 | 📦 Clone repo & cài packages | ~2 phút |
| 3 | 🦟 Khởi động MQTT broker | 30 giây |
| 4 | 🚀 Khởi động backend | 30 giây |
| 5 | 🌐 Tạo public URL (Cloudflare — không cần tài khoản!) | 20 giây |
| 6 | 📊 Mở dashboard & enjoy! | — |

## Bước 1 — 🔑 Điền Gemini API Key

Lấy **miễn phí** tại: https://aistudio.google.com/apikey  
*(Đăng nhập Google → Get API Key → Create API Key)*

In [ ]:
# ═══════════════════════════════════════════════════════════
# Dán API key Gemini của bạn vào đây
# ═══════════════════════════════════════════════════════════
GEMINI_API_KEY = "AIzaSy-xxxx-thay-bang-key-cua-ban"

# Email alert (tuỳ chọn — bỏ trống nếu không cần)
RESEND_API_KEY = ""
ALERT_TO = ""

# ── Validation ──────────────────────────────────────────────
if GEMINI_API_KEY.startswith("AIzaSy") and len(GEMINI_API_KEY) > 20:
    print(f"✅ Gemini API key: {GEMINI_API_KEY[:12]}...{GEMINI_API_KEY[-4:]}")
    print("   Tiếp tục bước 2!")
else:
    print("⚠️  Hãy điền Gemini API key thật vào ô trên!")
    print("   Lấy tại: https://aistudio.google.com/apikey")

## Bước 2 — 📦 Clone repo & cài dependencies

In [ ]:
import os, subprocess, sys

# Clone repo (branch cloud-classroom)
if not os.path.exists('/content/demo-iot'):
    print('📥 Cloning repo...')
    subprocess.run([
        'git', 'clone', 'https://github.com/hoamx2602/demo-iot.git',
        '/content/demo-iot', '--branch', 'cloud-classroom', '-q'
    ], check=True)
    print('✅ Clone xong!')
else:
    print('✅ Repo đã có, bỏ qua clone.')

os.chdir('/content/demo-iot')

print('📦 Cài Python packages...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', 'backend/requirements.txt'], check=True)

print('\n✅ Xong! Tiếp tục bước 3.')

## Bước 3 — 🦟 Khởi động Mosquitto MQTT Broker

In [ ]:
import subprocess, time

print('🦟 Cài Mosquitto...')
subprocess.run(['apt-get', 'install', '-y', '-q', 'mosquitto'],
               check=True, capture_output=True)

# Config cho phép kết nối ẩn danh
with open('/tmp/mosquitto.conf', 'w') as f:
    f.write('listener 1883\nallow_anonymous true\n')

# Dừng instance cũ nếu có
subprocess.run(['pkill', '-f', 'mosquitto'], capture_output=True)
time.sleep(0.5)

proc = subprocess.Popen(
    ['mosquitto', '-c', '/tmp/mosquitto.conf'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(1)

if proc.poll() is None:
    print(f'✅ Mosquitto đang chạy (PID {proc.pid}) — port 1883')
else:
    print('❌ Lỗi khởi động Mosquitto')

## Bước 4 — 🚀 Khởi động FastAPI Backend

In [ ]:
import subprocess, time, os, requests

os.chdir('/content/demo-iot')

# Tạo .env
with open('backend/.env', 'w') as f:
    f.write(f"""AI_PROVIDER=gemini
GEMINI_API_KEY={GEMINI_API_KEY}
MQTT_HOST=localhost
MQTT_PORT=1883
RESEND_API_KEY={RESEND_API_KEY}
ALERT_TO={ALERT_TO}
""")
print('✅ .env đã tạo')

# Dừng instance cũ
subprocess.run(['pkill', '-f', 'uvicorn'], capture_output=True)
time.sleep(1)

# Load env vào process environment
env_vars = {}
for line in open('backend/.env').read().strip().splitlines():
    if '=' in line and not line.startswith('#'):
        k, v = line.split('=', 1)
        env_vars[k.strip()] = v.strip()

log_file = open('/tmp/backend.log', 'w')
proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'backend.server:app',
     '--host', '0.0.0.0', '--port', '8000'],
    stdout=log_file, stderr=log_file,
    env={**os.environ, **env_vars}
)

print('⏳ Chờ backend khởi động', end='')
for _ in range(20):
    time.sleep(1)
    try:
        r = requests.get('http://localhost:8000/health', timeout=2)
        if r.status_code == 200:
            data = r.json()
            print(f'\n✅ Backend running! (PID {proc.pid})')
            print(f'   AI: {data.get("ai_provider")} | Key: {"✅ OK" if data.get("api_key_configured") else "❌ Chưa set — kiểm tra lại Bước 1"}')
            break
    except:
        print('.', end='', flush=True)
else:
    print('\n❌ Backend không lên. Log:')
    print(open('/tmp/backend.log').read()[-3000:])

## Bước 5 — 🌐 Tạo Public URL (Cloudflare Tunnel)

> **Không cần tài khoản, không cần token!**  
> Cloudflare tự động tạo URL `*.trycloudflare.com` miễn phí.

In [ ]:
import subprocess, time, re, threading

# Cài cloudflared
print('📥 Tải cloudflared...')
subprocess.run([
    'wget', '-q', '-O', '/usr/local/bin/cloudflared',
    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
], check=True)
subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)
print('✅ cloudflared đã sẵn sàng')

# Kill tunnel cũ
subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(1)

# Khởi động tunnel — capture output để lấy URL
tunnel_log = open('/tmp/cloudflared.log', 'w')
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000',
     '--no-autoupdate'],
    stdout=tunnel_log, stderr=subprocess.STDOUT
)

# Đợi URL xuất hiện trong log
print('⏳ Đợi Cloudflare kết nối', end='')
public_url = None
for _ in range(30):
    time.sleep(2)
    try:
        log_content = open('/tmp/cloudflared.log').read()
        match = re.search(r'https://[\w-]+\.trycloudflare\.com', log_content)
        if match:
            public_url = match.group(0)
            break
    except:
        pass
    print('.', end='', flush=True)

print()
if public_url:
    print()
    print('=' * 62)
    print('🎉  PUMPGUARD AI ĐANG CHẠY!')
    print('=' * 62)
    print()
    print(f'🌐  Dashboard (share cho học viên):')
    print(f'    {public_url}/dashboard/')
    print()
    print(f'🔗  API Docs:   {public_url}/docs')
    print(f'❤️   Health:    {public_url}/health')
    print()
    print('=' * 62)
    print('ℹ️   URL này hoạt động miễn phí, không giới hạn thời gian')
    print('    (chỉ reset khi Runtime Colab bị ngắt)')
    print('=' * 62)
else:
    print('❌ Không lấy được URL. Xem log:')
    print(open('/tmp/cloudflared.log').read()[-2000:])

## Bước 6 (Tuỳ chọn) — 📊 Stream dữ liệu sensor thật

Cell này stream dữ liệu từ CSV lên dashboard (blocking — nhấn ⏹ để dừng).  
**Nếu không có CSV**: dashboard vẫn dùng được với **Operator Controls** ở góc phải.

In [ ]:
import os
os.chdir('/content/demo-iot')

if not os.path.exists('data/sensor.csv'):
    print('⚠️  Không có data/sensor.csv')
    print('   → Dùng Operator Controls trên dashboard:')
    print('     ▶ Normal  |  ⚠ Simulate Anomaly  |  🔴 Simulate Critical')
    print('     Nút ⚙ ở góc phải màn hình dashboard')
else:
    print('▶ Bắt đầu replay... (Nhấn ⏹ để dừng)')
    print('-' * 50)
    !python scripts/mqtt_replay.py \
        --csv data/sensor.csv \
        --config data/sensor_groups.json \
        --start-at-anomaly \
        --compression 360

---
## 🛠 Troubleshooting

| Vấn đề | Giải pháp |
|--------|----------|
| Dashboard không kết nối WS | Chạy lại Bước 5 để lấy URL mới |
| AI hiện `[MOCK]` | Kiểm tra `GEMINI_API_KEY` ở Bước 1 |
| Colab Runtime timeout | Chạy lại từ Bước 3 (bỏ qua Bước 2) |
| Backend không lên | `!tail -30 /tmp/backend.log` |
| Cloudflare không kết nối | `!cat /tmp/cloudflared.log` |

### Xem log backend
```python
!tail -30 /tmp/backend.log
```
### Restart nhanh (sau khi Colab timeout)
Chạy lại **Bước 3 → 4 → 5** (bỏ qua Bước 2).